# Data Cleaning

Loads the raw dataset and applies the cleaning steps defined in `src/data/preprocess.py`.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
import os
os.chdir(project_root)

In [2]:
from src.data.load_data import load_config, load_raw_data
from src.data.preprocess import clean_data, describe_data, save_processed_data

config = load_config()
raw = load_raw_data(config)
print(raw.shape)
raw.head()

(1848, 7)


,Agency,Neighborhood,Price,link,sq_mtrs,Bedrooms,Bathrooms
0,Buy Rent Shelters,"General Mathenge, Westlands","KSh 155,000",/listings/4-bedroom-apartment-for-rent-general...,4.0,4.0,4.0
1,Kenya Classic Homes,"Kilimani, Dagoretti North","KSh 100,000",/listings/3-bedroom-apartment-for-rent-kiliman...,300.0,3.0,4.0
2,Absolute Estate Agents,"Hatheru Rd,, Lavington, Dagoretti North","KSh 75,000",/listings/3-bedroom-apartment-for-rent-lavingt...,3.0,3.0,5.0
3,A1 Properties Limited,"Kilimani, Dagoretti North","KSh 135,000",/listings/3-bedroom-apartment-for-rent-kiliman...,227.0,3.0,4.0
4,Pmc Estates Limited,"Imara Daima, Embakasi","KSh 50,000",/listings/3-bedroom-apartment-for-rent-imara-d...,3.0,3.0,NaN


1,848 raw listings, 7 columns. `Agency` and `link` aren't useful for price analysis, so they're dropped. `sq_mtrs` is also dropped - it contains implausibly small values for a property size (e.g. single-digit "square meters"), which are almost certainly data entry or scraping errors rather than genuine measurements. `Bedrooms` is used as the size indicator instead, since it's a more reliable signal in this dataset.

## Clean the data

In [3]:
clean = clean_data(raw)
print(clean.shape)
clean.head()

(1557, 4)


,Price_Ksh,Bedrooms,Bathrooms,Estate
0,155000.0,4,4,Westlands
1,100000.0,3,4,Dagoretti North
2,75000.0,3,5,Dagoretti North
3,135000.0,3,4,Dagoretti North
6,100000.0,2,3,Dagoretti North


1,557 rows remain after dropping rows with missing values (down from 1,848 — mostly driven by `Bathrooms`, which had 291 nulls). The dataset is now down to 4 columns: `Price_Ksh` (parsed from strings like `"KSh 100,000"`), `Bedrooms`, `Bathrooms`, and `Estate` (derived from the last segment of the original `Neighborhood` field).

In [4]:
describe_data(clean)

,Price_Ksh,Bedrooms,Bathrooms
count,1557.000000,1557.000000,1557.000000
mean,98339.751445,2.597303,2.595376
std,40175.785665,0.808259,1.000349
min,12000.000000,0.000000,1.000000
25%,70000.000000,2.000000,2.000000
50%,95000.000000,3.000000,2.000000
75%,130000.000000,3.000000,3.000000
max,240000.000000,6.000000,6.000000


Roughly half of listings are priced above KSh 95,000 (median), and the mean (~KSh 98,340) sits close to the median - the price distribution isn't wildly skewed. Half of listings have 3+ bedrooms and 2+ bathrooms, indicating the market leans toward larger, multi-room properties rather than studios or single-room units.

## Save the processed dataset

Persist the cleaned dataset to `data/processed/`, so downstream notebooks and pipeline stages can load it directly instead of repeating these steps.

In [5]:
save_processed_data(clean, config)